# Recurso Hídrico

> **Contexto de dominio:** [`docs/fuentes/recurso_hidrico.md`](../../docs/fuentes/recurso_hidrico.md)  
> **Bloque:** A | **Línea:** `recurso_hidrico`  
> **Variable principal:** `od` (mg/L)  
> **Modelos sugeridos:** SARIMA, SARIMAX, XGBOOST, RANDOM_FOREST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/recurso_hidrico.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.config import ENSO_LAG_MESES
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "recurso_hidrico"
VARIABLE = "od"
UNIDAD = "mg/L"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `recurso_hidrico` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "recurso_hidrico.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/recurso_hidrico.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/recurso_hidrico.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "od": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `recurso_hidrico` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_recurso_hidrico.html",
       title="EDA — Recurso Hídrico", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "od", title="Recurso Hídrico — od (mg/L)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "od", period="month")
plt.show()

<!-- ENRICHMENT: recurso_hidrico -->

## 3c. Capacidad de asimilacion y modelo de calidad QUAL2K

La **capacidad de asimilacion** es la cantidad de carga contaminante que un rio puede recibir sin superar los objetivos de calidad del PORH (Res. 0751/2018).

El modelo **Streeter-Phelps** (base de QUAL2K/QUAL2Kw) describe el deficit de OD:

```
DBO(x) = DBO0 * exp(-Kd * x/U)            Kd = tasa descomposicion (1/dia)
OD_deficit(x) = Dc * exp(-Ka * x/U)       Ka = tasa reaireacion (1/dia)
               - (Kd*DBO0)/(Ka-Kd) * [exp(-Kd*x/U) - exp(-Ka*x/U)]
Donde U = velocidad media del rio (m/s), x = distancia aguas abajo (km)
```

**QUAL2K** extiende este modelo con temperatura, nitrificacion, algas y sedimentos. Implementacion Python: `qual2kpy` o manual con ODE de scipy.

> **BMWP-Col (macroinvertebrados):** bioindicador biologico obligatorio en el diagnostico del PORH, complementa el ICA fisicoquimico con informacion de la calidad ecologica del agua.

In [ ]:
# Modelo Streeter-Phelps simplificado — perfil DBO y deficit de OD aguas abajo
# Simula la capacidad de asimilacion de un rio receptor de vertimiento

# Parametros del cuerpo de agua (rio andino tipico)
Q_rio = 5.0       # m3/s caudal receptor
U_vel = 0.4       # m/s velocidad media
OD_sat = 8.5      # mg/L saturacion OD (a ~1500 msnm)
OD_ini = 6.2      # mg/L OD aguas arriba del vertimiento
Kd = 0.25         # 1/dia tasa descomposicion DBO
Ka = 0.80         # 1/dia tasa reaireacion (K2)

# Vertimiento (punto de descarga PTAR municipal)
Q_vert = 0.15     # m3/s caudal vertido
DBO_vert = 180    # mg/L DBO5 del vertimiento
OD_vert = 1.0     # mg/L OD del vertimiento

# Mezcla inmediata aguas abajo del punto de descarga
DBO_mix = (Q_rio * 2.0 + Q_vert * DBO_vert) / (Q_rio + Q_vert)  # 2.0 mg/L DBO rio
OD_mix  = (Q_rio * OD_ini + Q_vert * OD_vert) / (Q_rio + Q_vert)
Dc_ini  = OD_sat - OD_mix  # deficit inicial

# Perfil a lo largo del rio (0 a 80 km)
x_km = np.linspace(0, 80, 200)
t = x_km * 1000 / (U_vel * 86400)  # dias de viaje

DBO_x = DBO_mix * np.exp(-Kd * t)
def_od = (Dc_ini * np.exp(-Ka * t)
          - (Kd * DBO_mix) / (Ka - Kd) * (np.exp(-Kd * t) - np.exp(-Ka * t)))
OD_x = np.clip(OD_sat - def_od, 0, OD_sat)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(x_km, DBO_x, lw=2, color='#e74c3c', label='DBO5 (mg/L)')
ax1.axhline(5, color='orange', ls='--', lw=1.5, label='Objetivo calidad PORH DBO=5')
x_meta_dbo = x_km[DBO_x < 5][0] if (DBO_x < 5).any() else 80
ax1.axvline(x_meta_dbo, color='gray', ls=':', lw=1)
ax1.set_xlabel('Distancia aguas abajo (km)'); ax1.set_ylabel('DBO5 (mg/L)')
ax1.set_title('Perfil DBO — Streeter-Phelps (base QUAL2K)', fontweight='bold')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

ax2.plot(x_km, OD_x, lw=2, color='#2980b9', label='OD (mg/L)')
ax2.axhline(4.0, color='red', ls='--', lw=1.5, label='Minimo OD uso piscicola=4 mg/L')
ax2.axhline(OD_sat, color='green', ls=':', lw=1, label=f'OD saturacion={OD_sat}')
od_min_km = x_km[OD_x.argmin()]
ax2.axvline(od_min_km, color='gray', ls=':', lw=1, label=f'Minimo OD en {od_min_km:.1f} km')
ax2.set_xlabel('Distancia aguas abajo (km)'); ax2.set_ylabel('OD (mg/L)')
ax2.set_title('Deficit de OD — Curva de sagging (capacidad asimilacion)', fontweight='bold')
ax2.legend(fontsize=7); ax2.grid(alpha=0.3)

plt.suptitle('Modelo QUAL2K simplificado — PORH (Res. 0751/2018 MADS)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

od_min = OD_x.min()
print(f'OD minimo: {od_min:.2f} mg/L a {od_min_km:.1f} km del vertimiento')
print(f'DBO5 cumple objetivo (< 5 mg/L) a partir de: {x_meta_dbo:.1f} km')
print(f'Capacidad asimilacion: Q_rio={Q_rio} m3/s puede asimilar ~{Q_vert*DBO_vert:.0f} g/s DBO')
print('BMWP-Col recomendado en km 0, 10, 30 y 60 para validar el modelo')

## 3b. Covariable ENSO (ONI)
> Lag recomendado para `recurso_hidrico` definido en `config.ENSO_LAG_MESES`.

In [ ]:
# --- Covariable ENSO (lag específico para Recurso Hídrico) ---
# Guard (M-20): avisar si LINEA no tiene lag específico — se aplicará el default.
if LINEA not in ENSO_LAG_MESES:
    _lag_default = ENSO_LAG_MESES["default"]
    print(f"[aviso] '{LINEA}' sin lag específico en ENSO_LAG_MESES; "
          f"se usará default ({_lag_default} meses).")
else:
    print(f"[ok] ENSO lag para '{LINEA}' = {ENSO_LAG_MESES[LINEA]} meses")

oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica=LINEA)
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["od"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas
> Compara `od` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["od"], variable="od")
if rep.empty:
    print("Sin normas colombianas registradas para 'od'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["od"], method="linear")
print(f"Faltantes antes: {df["od"].isna().sum()} | "
      f"después: {df_clean["od"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["od"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "SARIMAX":      get_model("sarimax", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: recurso_hidrico -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Hidrología / calidad del agua

- **NSE / KGE primarias** (igual que oferta hídrica).
- **Calidad de agua multivariada:** ICA combina pH, OD, conductividad, turbiedad, DBO5, coliformes — no analizar variables aisladamente.
- **Lag ENSO = 3 meses** (calidad responde a precipitación con retardo).
- **Normas:** `NORMA_AGUA_POTABLE` (Res. 2115/2007), `NORMA_VERTIMIENTOS` (Res. 631/2015).

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** Recurso Hídrico (`recurso_hidrico`)
- **Variable analizada:** `od` (mg/L)
- **Modelos ejecutados:** SARIMA, SARIMAX, XGBOOST, RANDOM_FOREST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/recurso_hidrico.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.